# Case3 TCO — own, rent, or call

Scaffold for Case 3: five-year cost stack for four adaptation/hosting alternatives plus an agentic option; accuracy threshold; own-vs-rent breakeven.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ri128-rutgers/quality-management-fall2026/blob/main/case3_tco.ipynb) · Also opens in Azure ML Studio notebooks or any Jupyter; nothing here needs a GPU.

**How to use:** run every cell top to bottom once. Cells marked `# >>> YOUR WORK` are where you change inputs or add interpretation. Keep your AI-assistant transcript if you used one; the verification log asks what you checked.

In [ ]:
# Setup — works in Colab and locally
import os, sys, io, urllib.request
REPO = 'ri128-rutgers/quality-management-fall2026'
DATA_BASE = f'https://raw.githubusercontent.com/{REPO}/main/' if 'google.colab' in sys.modules or not os.path.exists('../data') else '../data/'
if 'google.colab' in sys.modules:
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/{REPO}/main/tvm.py', 'tvm.py')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tvm import *
pd.options.display.float_format = '{:,.2f}'.format
def load(name): return pd.read_csv(DATA_BASE + name)
print('data from', DATA_BASE)

In [ ]:
A = load('case3_data_assumptions.csv').set_index('parameter')['value']; pilot = load('case3_pilot_messages.csv')
print(pilot.describe().round(1).T)

In [ ]:
# >>> YOUR WORK: current published prices, with URL and access date, in this cell only
PRICE = {
  'hosted_input_per_M_tokens': None,   # $ per million input tokens  — source:
  'hosted_output_per_M_tokens': None,  # $ per million output tokens — source:
  'small_input_per_M_tokens': None, 'small_output_per_M_tokens': None,   # distilled/small model hosted price
  'accel_rental_per_hour': None,       # $/accelerator-hour on-demand — source:
  'accel_committed_discount': None,    # fraction, e.g. 0.35
  'accel_purchase_per_unit': None,     # $ per accelerator incl. host share — source:
}
assert all(v is not None for v in PRICE.values()), 'fill in every price with a citation before running'

In [ ]:
N = 5; r = float(A['MARR']); msgs = [float(A['messages_year1'])*(1+float(A['message_growth_per_year']))**y for y in range(N)]
tin, tout = float(A['avg_input_tokens_per_message']), float(A['avg_output_tokens_per_message'])
fte = float(A['fte_loaded_usd']); evalr = float(A['evaluation_recurring_usd_per_year'])
def per_msg_hosted(extra_in=0, price_in=None, price_out=None):
    return ((tin+extra_in)*(price_in or PRICE['hosted_input_per_M_tokens']) + tout*(price_out or PRICE['hosted_output_per_M_tokens']))/1e6
# A prompt-only, B RAG, D distilled+router+cache, C fine-tune on rented accelerators
esc = 0.33; cache = float(A['alt_D_cache_hit_rate'])   # >>> YOUR WORK: choose inside the stated ranges
stack = {
 'A_api':   {'dev': 0.5*float(A['alt_C_eval_harness_build_usd']), 'per_msg': per_msg_hosted(), 'staff': 0.5*fte},
 'B_rag':   {'dev': 0.5*float(A['alt_C_eval_harness_build_usd']) + 80_000, 'per_msg': per_msg_hosted(float(A['rag_added_input_tokens'])), 'staff': 0.5*fte},
 'C_peft':  {'dev': float(A['alt_C_finetune_data_labeling_usd']) + float(A['alt_C_eval_harness_build_usd']) + float(A['alt_C_finetune_compute_accelerator_hours'])*PRICE['accel_rental_per_hour'],
             'per_msg': float(A['alt_C_inference_accelerator_hours_per_1M_messages'])*PRICE['accel_rental_per_hour']*(1-PRICE['accel_committed_discount'])/1e6, 'staff': 1.5*fte},
 'D_distill': {'dev': float(A['alt_D_distillation_engineering_usd']) + float(A['alt_C_eval_harness_build_usd']),
             'per_msg': (1-cache)*((1-esc)*per_msg_hosted(0, PRICE['small_input_per_M_tokens'], PRICE['small_output_per_M_tokens']) + esc*per_msg_hosted()), 'staff': 1.5*fte},
}
rows = []
for k, v in stack.items():
    cf = [-v['dev']] + [-(m*v['per_msg'] + v['staff'] + evalr) for m in msgs]
    rows.append({'alt': k, 'PW_5yr': -pw(cf, r), 'cost_per_1k_msgs_y1': 1000*v['per_msg'] + 1000*(v['staff']+evalr)/msgs[0], 'cost_per_1k_msgs_y5': 1000*v['per_msg'] + 1000*(v['staff']+evalr)/msgs[-1]})
print(pd.DataFrame(rows).set_index('alt').round(0))

In [ ]:
# Accuracy threshold: at what escalation share does D stop being cheapest? (escalation rises as the small model's approval rate falls)
for esc_try in np.arange(0.15, 0.7, 0.05):
    pm = (1-cache)*((1-esc_try)*per_msg_hosted(0, PRICE['small_input_per_M_tokens'], PRICE['small_output_per_M_tokens']) + esc_try*per_msg_hosted())
    cf = [-stack['D_distill']['dev']] + [-(m*pm + 1.5*fte + evalr) for m in msgs]
    print(round(esc_try,2), round(-pw(cf, r)))
# >>> YOUR WORK: relate escalation share to the small model's approval rate and the 0.86 floor

In [ ]:
# Own vs rent for C: breakeven utilization
hours_needed = [float(A['alt_C_inference_accelerator_hours_per_1M_messages'])*m/1e6 for m in msgs]
rent = -pw([0] + [-h*PRICE['accel_rental_per_hour']*(1-PRICE['accel_committed_discount']) for h in hours_needed], r)
for util in (0.2, 0.4, 0.6, 0.8):
    units = max(hours_needed)/(8760*util)
    own = units*PRICE['accel_purchase_per_unit'] + pw([0]+[-(units*PRICE['accel_purchase_per_unit']*0.18)]*N, r)*-1   # 18%/yr hosting+power+staff share; >>> YOUR WORK: justify
    print(f'utilization {util:.0%}: own {own:,.0f}  rent {rent:,.0f}')

In [ ]:
# Agentic option E: token cost per task is a distribution, plus a human checkpoint
calls = pilot['agentic_tool_calls'] + pilot['agentic_retries']
tokens_per_task = tin + calls*450 + tout*1.6     # 450 tokens per tool round-trip; >>> YOUR WORK: justify
cost_per_task = (tokens_per_task*PRICE['hosted_input_per_M_tokens']/1e6) + float(A['alt_E_human_checkpoint_seconds'])/3600*float(A['agent_loaded_hourly_usd'])
print('E cost per task: mean', round(cost_per_task.mean(),4), ' P90', round(cost_per_task.quantile(0.9),4))
cost_per_task.plot.hist(bins=40); plt.title('Cost per agentic task'); plt.show()